<a href="https://colab.research.google.com/github/Kavishka2401/CustomerChurnPredictionSystem/blob/master/2_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Data Pre-Processing for Neural Networks**

In [ ]:
# Mount the google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Provide path
data=pd.read_csv('/content/drive/MyDrive/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Make a copy of the original dataset to work with
data_copy = data.copy()
df_nn = data_copy

In [ ]:
# Create a numeric column for TotalCharges
df_nn['TotalCharges_numeric'] = pd.to_numeric(df_nn['TotalCharges'], errors='coerce')

# Check for missing values after conversion
missing_total_charges = df_nn['TotalCharges_numeric'].isnull().sum()
print(f"Missing values in TotalCharges_numeric: {missing_total_charges}")


In [ ]:
# Display rows with missing TotalCharges_numeric
missing_rows = df_nn[df_nn['TotalCharges_numeric'].isnull()]
missing_rows

In [ ]:
# Impute median for missing values
df_nn['TotalCharges_numeric'] = df_nn['TotalCharges_numeric'].fillna(median_total_charges)

In [ ]:
# Check if imputation has worked
missing_rows = df_nn[df_nn['TotalCharges_numeric'].isnull()]
missing_rows

In [ ]:
# IQR method to detect outliers
num_features = ['TotalCharges_numeric', 'MonthlyCharges', 'tenure']

for col in num_features:
    Q1 = df_nn[col].quantile(0.25)
    Q3 = df_nn[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df_nn[(df_nn[col] < lower_bound) | (df_nn[col] > upper_bound)]
    print(f"{col}: {len(outliers)} outliers")

In [ ]:
# Checking if skew handling should be done to the numerical columns
df_nn[['TotalCharges_numeric', 'MonthlyCharges', 'tenure']].skew()

In [ ]:
# Log transform TotalCharges_numeric (Skew handling)
df_nn['TotalCharges_log'] = np.log1p(df_nn['TotalCharges_numeric'])

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# List of numerical features to scale
num_features = ['TotalCharges_log', 'MonthlyCharges', 'tenure']

# Initialize MinMaxScaler
scaler = MinMaxScaler()

# Fit the scaler on the numerical features and transform
df_nn[num_features] = scaler.fit_transform(df_nn[num_features])

# Check the result
print(df_nn[num_features].describe())

In [ ]:
# List of categorical features
cat_features = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
                'PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies',
                'Contract', 'PaperlessBilling', 'PaymentMethod']

# One-hot encoding
df_nn_encoded = pd.get_dummies(df_nn, columns=cat_features, drop_first=True)

# Convert all boolean columns (True/False) to 0/1
bool_cols = df_nn_encoded.select_dtypes(include='bool').columns
df_nn_encoded[bool_cols] = df_nn_encoded[bool_cols].astype(int)

# Encode target variable Churn: No -> 0, Yes -> 1
df_nn_encoded['Churn'] = df_nn_encoded['Churn'].map({'No': 0, 'Yes': 1})

# Check the shape after encoding
print(df_nn_encoded.shape)

# Check first few rows
df_nn_encoded.head()

In [ ]:
# Feature Engineering

# Average monthly charge per tenure
# Adding 1 to tenure to avoid division by zero
df_nn_encoded['AvgChargePerMonth'] = df_nn_encoded['TotalCharges_numeric'] / (df_nn_encoded['tenure'] + 1)

# Service usage count
# List of columns representing optional services (encoded as 0/1)
service_cols = ['OnlineSecurity_Yes', 'OnlineBackup_Yes', 'DeviceProtection_Yes',
                'TechSupport_Yes', 'StreamingTV_Yes', 'StreamingMovies_Yes']

# Sum the services used by each customer
df_nn_encoded['ServiceCount'] = df_nn_encoded[service_cols].sum(axis=1)

# Check the first few rows to confirm
df_nn_encoded[['AvgChargePerMonth', 'ServiceCount']].head()


In [ ]:
# Features (X) and target (y)
X = df_nn_encoded.drop(columns=['Churn', 'customerID','TotalCharges','TotalCharges_numeric'])  # Drop target & ID
y = df_nn_encoded['Churn'].map({'No':0, 'Yes':1})       # Encode target as 0/1

# Train/Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Original training set shape: {X_train.shape}, {y_train.shape}")
print(f"Original class distribution:\n{y_train.value_counts()}")

# Apply SMOTE
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

print(f"SMOTE training set shape: {X_train_sm.shape}, {y_train_sm.shape}")
print(f"SMOTE class distribution:\n{y_train_sm.value_counts()}")

# Compute Class Weights
from sklearn.utils import class_weight
import numpy as np

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_sm),
    y=y_train_sm
)

class_weight_dict = dict(zip(np.unique(y_train_sm), class_weights))
print(f"Class weights to be used for NN training: {class_weight_dict}")

In [ ]:
# WITH TWO NEW FEATURES WE GOT FROM FEATURE ENGINEERING
# ------------------------------
# 1. FEATURE ENGINEERING
# ------------------------------

# Avg monthly charge per tenure (avoid division by zero)
df_nn_encoded['AvgChargePerMonth'] = df_nn_encoded['TotalCharges_numeric'] / (df_nn_encoded['tenure'] + 1)

# Service count (sum of optional services)
service_cols = [
    'OnlineSecurity_Yes', 'OnlineBackup_Yes', 'DeviceProtection_Yes',
    'TechSupport_Yes', 'StreamingTV_Yes', 'StreamingMovies_Yes'
]

df_nn_encoded['ServiceCount'] = df_nn_encoded[service_cols].sum(axis=1)


# ------------------------------
# 2. FEATURES AND TARGET
# ------------------------------

# X = all features except Churn & ID
X = df_nn_encoded.drop(columns=['Churn', 'customerID','TotalCharges','TotalCharges_numeric'])
y = df_nn_encoded['Churn'].map({'No':0, 'Yes':1})


# ------------------------------
# 3. TRAIN/TEST SPLIT
# ------------------------------
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Original training set shape: {X_train.shape}, {y_train.shape}")
print("Original class distribution:")
print(y_train.value_counts())


# ------------------------------
# 4. APPLY SMOTE (ONLY ON TRAINING SET)
# ------------------------------
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

print(f"\nSMOTE training set shape: {X_train_sm.shape}, {y_train_sm.shape}")
print("SMOTE class distribution:")
print(y_train_sm.value_counts())


# ------------------------------
# 5. CLASS WEIGHTS
# ------------------------------
from sklearn.utils import class_weight
import numpy as np

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_sm),
    y=y_train_sm
)

class_weight_dict = dict(zip(np.unique(y_train_sm), class_weights))
print("\nClass weights to be used for NN training:")
print(class_weight_dict)